# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

This playbook turns the Week-6 model into a ranked, human-reviewed action queue. I deliberately use the **honest, pre-label-window features from the Week-6 fix** (AUC 0.660 under a client-grouped split), not the Week-5 features, because those overlapped the label's own windows. All claims below use the observed / measured / decision-support language from the claim ladder, not causal language.

In [1]:
import json, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# Honest, pre-label-window features only (the Week-6 fix): the 90d windows overlap the label,
# so every feature below is rebuilt from the ONE 30-day block that sits before both label windows.
for base in ["impressions", "clicks", "sessions"]:
    df[f"{base}_old30"] = (df[f"{base}_90d"] - df[f"{base}_last_30d"] - df[f"{base}_prev_30d"]).clip(lower=0)
    df[f"log_{base}_old30"] = np.log1p(df[f"{base}_old30"])
df["ctr_old30"] = np.where(df["impressions_old30"] > 0, df["clicks_old30"] / df["impressions_old30"] * 100, np.nan)

PRE_LABEL_NUM = ["search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days",
                  "days_since_last_update", "log_impressions_old30", "log_clicks_old30", "log_sessions_old30", "ctr_old30"]
CAT = ["competition_level", "content_type", "main_intent"]
X = df[PRE_LABEL_NUM + CAT]
y = df["is_declining"].values
groups = df["client_id"].values

def preprocess():
    num = make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler())
    cat = make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"), OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("num", num, PRE_LABEL_NUM), ("cat", cat, CAT)])

# Out-of-fold scores: every page is scored by a model that never saw its client (Week-6 split).
folds = list(GroupKFold(n_splits=5).split(X, y, groups))
df["risk_score"] = 0.0
for tr, te in folds:
    m = make_pipeline(preprocess(), LogisticRegression(max_iter=3000)).fit(X.iloc[tr], y[tr])
    df.loc[df.index[te], "risk_score"] = m.predict_proba(X.iloc[te])[:, 1]

auc = roc_auc_score(y, df["risk_score"])
print(f"out-of-fold ROC AUC: {auc:.3f} | base rate: {y.mean():.3f} | rows: {len(df):,} | clients: {df['client_id'].nunique()}")

out-of-fold ROC AUC: 0.652 | base rate: 0.542 | rows: 30,000 | clients: 32


**Archetype to action mapping.** Five reason codes, each a plain combination of visibility, position, CTR and staleness, each mapped to one action. `other` is pages with a signal too thin to act on (not enough old-window traffic, or missing position data) and gets no action, not a guess.

In [2]:
VISIBLE   = df["impressions_old30"] >= 100          # old30 covers ~1/3 of a 90-day window, so the bar is lower than Week 4's 500
PAGE_ONE  = (df["avg_position"] > 0) & (df["avg_position"] <= 10)
LOW_CTR   = df["ctr_old30"] <= 0.12
STALE     = df["days_since_last_update"] >= 90
FRESH     = df["days_since_last_update"] < 30

df["reason_code"] = np.select(
    [VISIBLE & PAGE_ONE & LOW_CTR & STALE, VISIBLE & PAGE_ONE & LOW_CTR, VISIBLE & STALE, FRESH],
    ["ctr_gap_and_stale", "ctr_gap_page1", "stale_only", "fresh"],
    default="other",
)
df["action_label"] = df["reason_code"].map({
    "ctr_gap_and_stale": "Refresh content AND rewrite title/meta for CTR",
    "ctr_gap_page1":     "Rewrite title/meta for CTR",
    "stale_only":        "Refresh content",
    "fresh":             "No action -- recently updated",
    "other":             "No action -- insufficient signal",
})

archetype_table = (df.groupby("reason_code")
                      .agg(pages=("reason_code", "size"), mean_risk_score=("risk_score", "mean"), observed_declining_rate=("is_declining", "mean"))
                      .round(3).sort_values("mean_risk_score", ascending=False))
archetype_table

,pages,mean_risk_score,observed_declining_rate
reason_code,,,
ctr_gap_and_stale,1216,0.754,0.747
ctr_gap_page1,1787,0.685,0.711
stale_only,6290,0.684,0.607
fresh,18713,0.528,0.492
other,1994,0.513,0.527


**Does the ranking beat random?** The base rate is 0.542. The top 50 of 30,000 pages are 88.0% in the declining proxy, falling to 77.4% by the top 500 and 75.5% by the top 1,000. All numbers are out-of-fold, from a model that never saw that page's client.

In [3]:
queue = (df.sort_values(["risk_score", "ctr_old30", "days_since_last_update"], ascending=[False, True, False], kind="stable")
           .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)

def precision_at_k(k):
    return queue["is_declining"].head(k).mean()

pd.DataFrame({"top K": [50, 100, 250, 500, 1000], "observed declining rate in top K": [round(precision_at_k(k), 3) for k in [50, 100, 250, 500, 1000]]})

,top K,observed declining rate in top K
0,50,0.880
1,100,0.860
2,250,0.816
3,500,0.774
4,1000,0.755


In [4]:
cols = ["rank", "client_id", "content_id", "risk_score", "reason_code", "action_label"]
queue[cols].head(10)

,rank,client_id,content_id,risk_score,reason_code,action_label
0,1,client_6208ef0f77,content_d4786a3f7544,0.946385,stale_only,Refresh content
1,2,client_6208ef0f77,content_b2ec40b9d68c,0.935507,stale_only,Refresh content
2,3,client_6208ef0f77,content_e454bc31017a,0.932764,stale_only,Refresh content
3,4,client_6208ef0f77,content_ec5e5f49929b,0.931558,stale_only,Refresh content
4,5,client_6208ef0f77,content_7760048e1609,0.926033,stale_only,Refresh content
5,6,client_6208ef0f77,content_d8fe64aaac6b,0.925825,stale_only,Refresh content
6,7,client_6208ef0f77,content_39b6f6812582,0.925275,fresh,No action -- recently updated
7,8,client_6208ef0f77,content_8f16c82296ab,0.921819,stale_only,Refresh content
8,9,client_19581e27de,content_fb5ea6f6ea7c,0.921670,ctr_gap_and_stale,Refresh content AND rewrite title/meta for CTR
9,10,client_6208ef0f77,content_fb4bf6555c79,0.921072,stale_only,Refresh content


**Reading the top 10 honestly:** 9 of the top 10 rows belong to one client (`client_6208ef0f77`). That is expected given how risk score is computed; it is also the first thing the no-go section below has to address, not something to hide by reshuffling the display.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who this is for:** a content editor or SEO lead deciding which pages to look at first in a weekly or monthly review cycle, across the 32 clients in this snapshot.

**What it is:** a ranked queue, ordered by an out-of-fold risk score, with a plain-English reason code per page.

**What it is not:** a forecast of what will happen to any single page, a page-by-page verdict, or a tool that decides what to publish. The label it's trained against, `is_declining`, is a same-month proxy (second half of a 90-day window against the first), not a future outcome.

**Where it stops being valid:**
- **Outside this one snapshot month.** The model has been checked on one month of data for 32 clients. It has not been tested on the sealed final month or on a client it has never seen in a meaningful volume.
- **For a client with very few pages.** Per-client page counts range from 3 to 7,008 (median 567); a handful of pages per client isn't enough to trust a ranking within that client.
- **As a one-page verdict.** The precision numbers above describe the queue in aggregate, not any single row's correctness.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before anyone acts on a row, a human checks:**
- The reason code against the page itself: does low CTR or staleness actually look true on the live page, not just in the table?
- Whether the page's topic is still relevant to the business (a stale page about a discontinued product should not be "refreshed").
- Cost vs. value: a `ctr_gap_and_stale` page with high search volume is worth an editor's hour; an `other` or low-volume `stale_only` page usually is not. I did not build a cost model here, because I have no cost or editor-hours data; this stays a judgment call per client.

**What should NOT be automated:**
- **Auto-publishing any rewritten title, meta description or content change.** The model flags a page; it does not write or approve copy.
- **Auto-deprioritizing or hiding pages scored `fresh` or `other`.** Those are "no evidence to act", not "confirmed healthy".
- **Treating the queue order as a guarantee.** Two pages four ranks apart can be a coin flip apart in reality; the ranking is useful in blocks of fifty or a hundred, not row by row.
- **Running this unreviewed on a client it has not been checked against.** 9 of the top 10 rows here come from one client. Before using this for a new client, re-run the archetype table in section 1 for that client alone and sanity-check it with that client's editor.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**What would tell us this has gone stale.** I split this into a cheap weekly check and a monthly one, plus the conditions that call for a retrain, not just a re-check.

In [5]:
monitoring = {
    "weekly, cheap": [
        "Volume check: are the five reason-code buckets roughly the same size as this run (fresh ~62%, stale_only ~21%, other ~7%, ctr_gap_page1 ~6%, ctr_gap_and_stale ~4%)? A big shift means the input data or a client mix changed.",
        "Coverage check: is any one client still supplying more than half of the top 100? That was true in this run (one client was 78% of the top 100) and is the single most important thing to re-check before each use.",
    ],
    "monthly": [
        "Re-score last month's top 100 against this month's actual trend_direction: does the observed declining rate in the top 100 stay within about 10 points of 0.86 (this run)? A sustained drop below ~0.70 is a retrain trigger.",
        "Spot-check 5 acted-on pages and 5 skipped pages with an editor: did the action make sense in hindsight?",
    ],
    "retrain triggers (any one is enough)": [
        "Observed declining rate in the top 100 falls below 0.70 for two months running.",
        "A client's page count, topic mix, or CMS changes materially (new site section, migration, rebrand).",
        "The data dictionary's *_90d / *_last_30d / *_prev_30d definitions change -- the feature rebuild in section 1 depends on them exactly.",
        "A full quarter has passed: the model has only ever been checked on one snapshot month.",
    ],
}
monitoring

{'weekly, cheap': ['Volume check: are the five reason-code buckets roughly the same size as this run (fresh ~62%, stale_only ~21%, other ~7%, ctr_gap_page1 ~6%, ctr_gap_and_stale ~4%)? A big shift means the input data or a client mix changed.',
  'Coverage check: is any one client still supplying more than half of the top 100? That was true in this run (one client was 78% of the top 100) and is the single most important thing to re-check before each use.'],
 'monthly': ["Re-score last month's top 100 against this month's actual trend_direction: does the observed declining rate in the top 100 stay within about 10 points of 0.86 (this run)? A sustained drop below ~0.70 is a retrain trigger.",
  'Spot-check 5 acted-on pages and 5 skipped pages with an editor: did the action make sense in hindsight?'],
 'retrain triggers (any one is enough)': ['Observed declining rate in the top 100 falls below 0.70 for two months running.',
  "A client's page count, topic mix, or CMS changes materially (n

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Three files, for next week's paper to cite directly: the ranked queue, the archetype summary table, and one figure.

In [6]:
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

export_cols = ["rank", "client_id", "content_id", "risk_score", "reason_code", "action_label",
               "search_volume", "avg_position", "ctr_old30", "days_since_last_update", "content_age_days", "main_intent"]
queue[export_cols].to_csv("../outputs/w07_action_queue.csv", index=False)
print(f"wrote ../outputs/w07_action_queue.csv ({len(queue):,} rows)")

archetype_table.to_csv("../figures/w07_archetype_summary.csv")
print("wrote ../figures/w07_archetype_summary.csv")

wrote ../outputs/w07_action_queue.csv (30,000 rows)
wrote ../figures/w07_archetype_summary.csv


In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
order = archetype_table.index
ax.bar(order, archetype_table["observed_declining_rate"], color="#1F5F5B")
ax.axhline(y.mean(), color="#B45309", linestyle="--", label=f"base rate ({y.mean():.2f})")
ax.set_ylabel("observed declining rate")
ax.set_title("Observed declining rate by reason code (out-of-fold)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
plt.tight_layout()
plt.savefig("../figures/w07_archetype_declining_rate.png", dpi=150)
plt.close()
print("wrote ../figures/w07_archetype_declining_rate.png")

wrote ../figures/w07_archetype_declining_rate.png


In [8]:
metrics = {
    "task": "w07_action_playbook",
    "model": "Logistic Regression, pre-label features only (Week-6 fix), GroupKFold(5) by client_id",
    "n_rows": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "base_rate": round(float(y.mean()), 4),
    "out_of_fold_auc": round(float(auc), 4),
    "precision_at_k": {str(k): round(float(precision_at_k(k)), 4) for k in [50, 100, 250, 500, 1000]},
    "reason_code_counts": df["reason_code"].value_counts().to_dict(),
    "exports": ["work/outputs/w07_action_queue.csv", "work/figures/w07_archetype_summary.csv", "work/figures/w07_archetype_declining_rate.png"],
}
with open("../outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "task": "w07_action_playbook",
  "model": "Logistic Regression, pre-label features only (Week-6 fix), GroupKFold(5) by client_id",
  "n_rows": 30000,
  "n_clients": 32,
  "base_rate": 0.5421,
  "out_of_fold_auc": 0.6519,
  "precision_at_k": {
    "50": 0.88,
    "100": 0.86,
    "250": 0.816,
    "500": 0.774,
    "1000": 0.755
  },
  "reason_code_counts": {
    "fresh": 18713,
    "stale_only": 6290,
    "other": 1994,
    "ctr_gap_page1": 1787,
    "ctr_gap_and_stale": 1216
  },
  "exports": [
    "work/outputs/w07_action_queue.csv",
    "work/figures/w07_archetype_summary.csv",
    "work/figures/w07_archetype_declining_rate.png"
  ]
}


**Receipts for the paper.** `work/outputs/w07_action_queue.csv` is the full 30,000-row queue (git-ignored by design; the notebook regenerates it). `work/figures/w07_archetype_summary.csv` and `w07_archetype_declining_rate.png` are committed, and `work/outputs/w07_playbook_metrics.json` holds every number quoted in this notebook, so the paper's recommendations section can cite this file instead of retyping figures by hand.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.